# Healthcare Data Generator - Microsoft Fabric (100% Inlined)

This notebook contains the **complete healthcare synthetic data generator** with all logic inlined—no external dependencies on `src/` modules.

## Features
- ✅ Realistic hospital seasonality (pediatric +35%, cardiac +38%, cancer stable)
- ✅ Specialty-aware encounter generation and admission rates
- ✅ Payer market-share realism (Medicare, Medicaid, Commercial, Uninsured)
- ✅ ER occupancy modeling with hourly and day-of-week variation
- ✅ Weather-based diagnosis multipliers (flu in cold weather, falls in snow)
- ✅ Full rebuild from configurable start date to today
- ✅ Writes directly to Microsoft Fabric SQL via SQLAlchemy + pyodbc

## Quick Start
1. **Run Cell 1**: Load configuration and set `CONNECTION_STRING`
2. **Run Cell 2**: Validate environment (Python packages, ODBC drivers)
3. **Run Cell 3**: Execute the generator with your desired start date
4. **Run Cell 4**: Validate results and check row counts

## Configuration
- **CONNECTION_STRING**: Read from `src/config_local.py` or set environment variable
- **Rebuild Start Date**: Edit `rebuild_start` in Cell 3 (format: YYYY-MM-DD)
- **Export Parquet**: Set `export_parquet = True` in Cell 3 to also export data files

## Cell 1: Configuration & Connection Setup

In [ ]:
import os
import sys
import urllib.parse

# Add src folder to Python path (in case config_local.py is there)
repo_root = os.getcwd()
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f'Working directory: {repo_root}')
print(f'Python version: {sys.version}')

# Attempt to load CONNECTION_STRING from local config
CONNECTION_STRING = None
try:
    from config_local import CONNECTION_STRING as LOCAL_CONNECTION_STRING
    CONNECTION_STRING = LOCAL_CONNECTION_STRING
    print('✓ CONNECTION_STRING loaded from src/config_local.py')
except ImportError:
    # Try environment variable
    CONNECTION_STRING = os.getenv('CONNECTION_STRING')
    if CONNECTION_STRING:
        print('✓ CONNECTION_STRING loaded from environment variable')
    else:
        print('⚠ No CONNECTION_STRING found. Set via environment or create src/config_local.py')

# Convert JDBC format to ODBC if needed
if CONNECTION_STRING and CONNECTION_STRING.startswith('jdbc:sqlserver://'):
    print('ℹ Detected JDBC format connection string, converting to ODBC...')
    # Extract components from JDBC string
    jdbc_str = CONNECTION_STRING.replace('jdbc:sqlserver://', '')
    parts = jdbc_str.split(';')
    server_port = parts[0]
    params = {}
    for part in parts[1:]:
        if '=' in part:
            key, value = part.split('=', 1)
            params[key] = value
    
    # Build ODBC connection string
    driver = '{ODBC Driver 18 for SQL Server}'
    odbc_parts = [f'DRIVER={driver}', f'SERVER={server_port}']
    
    # Map common parameters
    if 'database' in params:
        odbc_parts.append(f"DATABASE={params['database']}")
    if 'encrypt' in params and params['encrypt'].lower() == 'true':
        odbc_parts.append('Encrypt=yes')
    if 'trustServerCertificate' in params and params['trustServerCertificate'].lower() == 'true':
        odbc_parts.append('TrustServerCertificate=yes')
    if 'authentication' in params:
        if params['authentication'] == 'ActiveDirectoryInteractive':
            odbc_parts.append('Authentication=ActiveDirectoryInteractive')
    
    CONNECTION_STRING = ';'.join(odbc_parts)
    print(f'✓ Converted to ODBC format')
    print(f'  Driver: ODBC Driver 18 for SQL Server')
    print(f'  Authentication: ActiveDirectoryInteractive')

if CONNECTION_STRING:
    print(f'  Connection string length: {len(CONNECTION_STRING)} characters')
else:
    print('  Generator will fail without CONNECTION_STRING—please set it before proceeding.')

## Cell 2: Environment Validation

In [ ]:
import importlib.util

print('\n[1/3] Checking required Python packages...')
required_packages = ['pandas', 'sqlalchemy', 'pyodbc', 'faker', 'requests']
missing = []

for pkg in required_packages:
    spec = importlib.util.find_spec(pkg)
    if spec is None:
        print(f'  ✗ {pkg:20s}: NOT FOUND')
        missing.append(pkg)
    else:
        print(f'  ✓ {pkg:20s}: available')

if missing:
    print(f'\n❌ Missing packages: {", ".join(missing)}')
    print('   Install them in your Fabric environment before proceeding.')
    print('   Run: pip install ' + ' '.join(missing))
else:
    print('\n✓ All required packages present\n')

print('[2/3] Checking ODBC drivers...')
try:
    import pyodbc
    drivers = pyodbc.drivers()
    print(f'  ✓ pyodbc version: {pyodbc.version}')
    print(f'  Available drivers: {len(drivers)}')
    for driver in drivers:
        if 'SQL Server' in driver:
            print(f'    ✓ {driver}')
except Exception as e:
    print(f'  ⚠ Could not list ODBC drivers: {e}')

print('\n[3/3] Testing database connectivity...')
if CONNECTION_STRING:
    try:
        from sqlalchemy import create_engine, inspect
        engine = create_engine(f'mssql+pyodbc:///?odbc_connect={CONNECTION_STRING}')
        inspector = inspect(engine)
        tables = inspector.get_table_names()
        print(f'  ✓ Connected to database')
        print(f'  Found {len(tables)} existing tables')
        if tables:
            print(f'    Sample tables: {", ".join(tables[:5])}')
    except Exception as e:
        print(f'  ✗ Connection error: {type(e).__name__}')
        print(f'    {e}')
else:
    print('  ⚠ Skipping—no CONNECTION_STRING set')

## Cell 3: All Generator Code (Inlined)

This cell contains the complete generator: all constants, helper functions, and the main orchestrator.
Run this cell to load all functions into memory.

In [ ]:
# ============================================================================
# HEALTHCARE DATA GENERATOR - COMPLETE INLINED IMPLEMENTATION
# ============================================================================

import pandas as pd
from faker import Faker
from sqlalchemy import create_engine, text, inspect
import random
from datetime import datetime, timedelta
import os
import requests
import json
import warnings
warnings.filterwarnings('ignore')

fake = Faker()

# ============================================================================
# CONFIGURATION & CONSTANTS
# ============================================================================

# Hourly encounter multiplier (0–23, midnight to 23:00)
HOURLY_MULTIPLIERS = [
    0.65, 0.54, 0.49, 0.43, 0.43, 0.60, 0.87, 1.03, 1.14, 1.14,
    1.14, 1.14, 1.08, 1.08, 1.14, 1.19, 1.30, 1.41, 1.52, 1.46,
    1.30, 1.14, 0.98, 0.81
]

# Day-of-week multiplier (Mon=0 → Sun=6)
DAY_OF_WEEK_MULTIPLIERS = [
    1.06,  # Monday
    0.98,  # Tuesday
    0.96,  # Wednesday
    0.96,  # Thursday
    1.00,  # Friday
    1.03,  # Saturday
    1.02   # Sunday
]

# Monthly multiplier (Jan=0 → Dec=11)
MONTHLY_MULTIPLIERS = [
    1.12, 1.08, 1.02, 0.95, 0.93, 0.88,  # Jan-Jun
    0.85, 0.88, 0.95, 0.98, 1.02, 1.12   # Jul-Dec
]

# Hospital-type-specific monthly multipliers (Jan-Dec)
PEDIATRIC_MONTHLY = [1.35, 1.28, 0.95, 0.82, 0.78, 0.75, 0.85, 0.88, 0.92, 0.98, 1.18, 1.26]
CARDIAC_MONTHLY = [1.38, 1.32, 1.08, 0.92, 0.85, 0.88, 0.87, 0.89, 0.95, 1.02, 1.18, 1.28]
CANCER_MONTHLY = [1.05, 1.03, 1.02, 1.01, 0.98, 0.95, 0.94, 0.96, 1.00, 1.02, 1.04, 1.06]
GENERAL_MONTHLY = [1.12, 1.08, 0.98, 0.92, 0.88, 0.85, 0.90, 0.93, 0.95, 1.00, 1.08, 1.18]
REHAB_MONTHLY = [1.04, 1.02, 1.08, 1.10, 1.01, 0.98, 0.92, 0.95, 1.05, 1.12, 1.06, 1.00]

# Hourly ED bed occupancy (% of licensed beds occupied, 0-23 hours)
HOURLY_ED_OCCUPANCY = [
    45, 42, 40, 38, 40, 45, 50, 58, 68, 75, 82, 85,  # Hours 0-11 (midnight-11am)
    88, 87, 88, 89, 88, 85, 82, 80, 78, 75, 70, 62   # Hours 12-23 (noon-11pm)
]

# Day-of-week occupancy variance
DAY_OF_WEEK_OCCUPANCY_MULT = {
    'Monday': 1.02, 'Tuesday': 1.01, 'Wednesday': 1.00,
    'Thursday': 1.02, 'Friday': 1.04, 'Saturday': 0.97, 'Sunday': 0.96
}

# Procedure weight seasonal adjustments
PROCEDURE_SEASONALITY = {
    'Blood Test': 1.00, 'X-Ray': 1.05, 'MRI Scan': 1.00, 'CT Scan': 1.03,
    'Colonoscopy': 0.95, 'Endoscopy': 0.98, 'Cardiac Catheterization': 1.15,
    'Appendectomy': 1.10, 'Hip Replacement': 1.08, 'Knee Arthroscopy': 1.06, 'Other': 1.00
}

# Payer market share
PAYER_MARKET_SHARE = {
    'Medicare': 0.36, 'Medicaid': 0.20, 'Commercial': 0.35,
    'Uninsured': 0.05, 'Other': 0.04
}

# Specialty-specific payer adjustments
SPECIALTY_PAYER_ADJUST = {
    'pediatric': {'Medicaid': 1.35, 'Commercial': 0.85, 'Medicare': 0.2},
    'cardiac': {'Medicare': 1.10, 'Commercial': 1.05, 'Medicaid': 0.9},
    'cancer': {'Commercial': 1.10, 'Medicare': 1.05, 'Medicaid': 0.9},
    'rehab': {'Medicare': 1.20, 'Medicaid': 0.95, 'Commercial': 0.9},
    'general': {}, 'specialty': {}
}

# Hospital definitions
HOSPITALS = [
    {'hospital_id': 1, 'name': 'Contoso Medical Center', 'bed_count': 750, 'specialty': 'general'},
    {'hospital_id': 2, 'name': 'Contoso University Hospital', 'bed_count': 650, 'specialty': 'general'},
    {'hospital_id': 3, 'name': 'Contoso General Hospital', 'bed_count': 550, 'specialty': 'general'},
    {'hospital_id': 4, 'name': 'Contoso Regional Hospital', 'bed_count': 400, 'specialty': 'general'},
    {'hospital_id': 5, 'name': 'Contoso Community Hospital', 'bed_count': 350, 'specialty': 'general'},
    {'hospital_id': 6, 'name': 'Contoso Heart Institute', 'bed_count': 280, 'specialty': 'cardiac'},
    {'hospital_id': 7, 'name': 'Contoso Children\'s Hospital', 'bed_count': 320, 'specialty': 'pediatric'},
    {'hospital_id': 8, 'name': 'Contoso Cancer Center', 'bed_count': 200, 'specialty': 'cancer'},
    {'hospital_id': 9, 'name': 'Contoso Rehab Center', 'bed_count': 180, 'specialty': 'rehab'},
    {'hospital_id': 10, 'name': 'Contoso Diagnostic Center', 'bed_count': 120, 'specialty': 'specialty'}
]

# Departments
DEPARTMENTS = [
    # Hospital 1
    {'department_id': 1, 'name': 'Emergency', 'hospital_id': 1, 'specialty_type': 'Emergency'},
    {'department_id': 2, 'name': 'Internal Medicine', 'hospital_id': 1, 'specialty_type': 'Medical'},
    {'department_id': 3, 'name': 'Surgery', 'hospital_id': 1, 'specialty_type': 'Surgical'},
    {'department_id': 4, 'name': 'Cardiology', 'hospital_id': 1, 'specialty_type': 'Medical'},
    {'department_id': 5, 'name': 'Radiology', 'hospital_id': 1, 'specialty_type': 'Diagnostic'},
    # Hospital 2
    {'department_id': 6, 'name': 'Emergency', 'hospital_id': 2, 'specialty_type': 'Emergency'},
    {'department_id': 7, 'name': 'Internal Medicine', 'hospital_id': 2, 'specialty_type': 'Medical'},
    {'department_id': 8, 'name': 'Trauma Surgery', 'hospital_id': 2, 'specialty_type': 'Surgical'},
    {'department_id': 9, 'name': 'Orthopedics', 'hospital_id': 2, 'specialty_type': 'Surgical'},
    {'department_id': 10, 'name': 'Neurology', 'hospital_id': 2, 'specialty_type': 'Medical'},
    # Hospital 3
    {'department_id': 11, 'name': 'Emergency', 'hospital_id': 3, 'specialty_type': 'Emergency'},
    {'department_id': 12, 'name': 'Internal Medicine', 'hospital_id': 3, 'specialty_type': 'Medical'},
    {'department_id': 13, 'name': 'Surgery', 'hospital_id': 3, 'specialty_type': 'Surgical'},
    {'department_id': 14, 'name': 'Radiology', 'hospital_id': 3, 'specialty_type': 'Diagnostic'},
    # Hospital 4
    {'department_id': 15, 'name': 'Emergency', 'hospital_id': 4, 'specialty_type': 'Emergency'},
    {'department_id': 16, 'name': 'Internal Medicine', 'hospital_id': 4, 'specialty_type': 'Medical'},
    {'department_id': 17, 'name': 'Surgery', 'hospital_id': 4, 'specialty_type': 'Surgical'},
    # Hospital 5
    {'department_id': 18, 'name': 'Emergency', 'hospital_id': 5, 'specialty_type': 'Emergency'},
    {'department_id': 19, 'name': 'Internal Medicine', 'hospital_id': 5, 'specialty_type': 'Medical'},
    {'department_id': 20, 'name': 'Surgery', 'hospital_id': 5, 'specialty_type': 'Surgical'},
    # Hospital 6 (Cardiac)
    {'department_id': 21, 'name': 'Cardiology', 'hospital_id': 6, 'specialty_type': 'Medical'},
    {'department_id': 22, 'name': 'Interventional Cardiology', 'hospital_id': 6, 'specialty_type': 'Surgical'},
    {'department_id': 23, 'name': 'Cardiac Surgery', 'hospital_id': 6, 'specialty_type': 'Surgical'},
    {'department_id': 24, 'name': 'Cardiac Rehabilitation', 'hospital_id': 6, 'specialty_type': 'Medical'},
    # Hospital 7 (Pediatric)
    {'department_id': 25, 'name': 'Pediatrics', 'hospital_id': 7, 'specialty_type': 'Medical'},
    {'department_id': 26, 'name': 'Pediatric Surgery', 'hospital_id': 7, 'specialty_type': 'Surgical'},
    {'department_id': 27, 'name': 'Pediatric ER', 'hospital_id': 7, 'specialty_type': 'Emergency'},
    {'department_id': 28, 'name': 'Neonatology', 'hospital_id': 7, 'specialty_type': 'Medical'},
    # Hospital 8 (Cancer)
    {'department_id': 29, 'name': 'Oncology', 'hospital_id': 8, 'specialty_type': 'Medical'},
    {'department_id': 30, 'name': 'Hematology', 'hospital_id': 8, 'specialty_type': 'Medical'},
    {'department_id': 31, 'name': 'Infusion Center', 'hospital_id': 8, 'specialty_type': 'Medical'},
    {'department_id': 32, 'name': 'Palliative Care', 'hospital_id': 8, 'specialty_type': 'Medical'},
    # Hospital 9 (Rehab)
    {'department_id': 33, 'name': 'Physical Rehabilitation', 'hospital_id': 9, 'specialty_type': 'Medical'},
    {'department_id': 34, 'name': 'Occupational Therapy', 'hospital_id': 9, 'specialty_type': 'Medical'},
    {'department_id': 35, 'name': 'Speech Therapy', 'hospital_id': 9, 'specialty_type': 'Medical'},
    # Hospital 10 (Diagnostic)
    {'department_id': 36, 'name': 'Radiology', 'hospital_id': 10, 'specialty_type': 'Diagnostic'},
    {'department_id': 37, 'name': 'Pathology', 'hospital_id': 10, 'specialty_type': 'Diagnostic'},
    {'department_id': 38, 'name': 'Laboratory', 'hospital_id': 10, 'specialty_type': 'Diagnostic'}
]

# Procedure weights
PROCEDURE_WEIGHTS = {
    'Blood Test': 0.25, 'X-Ray': 0.20, 'MRI Scan': 0.10, 'CT Scan': 0.08,
    'Colonoscopy': 0.07, 'Endoscopy': 0.06, 'Cardiac Catheterization': 0.05,
    'Appendectomy': 0.04, 'Hip Replacement': 0.03, 'Knee Arthroscopy': 0.02, 'Other': 0.10
}

# Department demand multipliers (real-world patient volume)
DEPARTMENT_DEMAND_MULTIPLIERS = {
    'Emergency': 2.5, 'Internal Medicine': 1.8, 'Surgery': 1.2, 'Cardiology': 1.5,
    'Orthopedics': 1.4, 'Radiology': 1.9, 'Oncology': 1.3, 'Pediatrics': 1.6,
    'Neurology': 1.1, 'Diagnostic': 1.7, 'Medical': 1.5, 'Surgical': 1.0,
    'Infusion Center': 0.9, 'Palliative Care': 0.6, 'Physical Rehabilitation': 0.8,
    'Occupational Therapy': 0.7, 'Speech Therapy': 0.5, 'Neonatology': 0.9,
    'Pathology': 1.2, 'Laboratory': 1.8
}

# Realistic department bed allocation percentages (% of total hospital beds)
DEPARTMENT_BED_ALLOCATION = {
    'Emergency': 0.06,
    'Internal Medicine': 0.35,
    'Surgery': 0.25,
    'Trauma Surgery': 0.20,
    'Cardiology': 0.15,
    'Orthopedics': 0.12,
    'Neurology': 0.10,
    'Radiology': 0.03,
    'Interventional Cardiology': 0.30,
    'Cardiac Surgery': 0.40,
    'Cardiac Rehabilitation': 0.25,
    'Pediatrics': 0.50,
    'Pediatric Surgery': 0.30,
    'Pediatric ER/Urgent Care': 0.05,
    'Neonatology': 0.15,
    'Oncology': 0.50,
    'Hematology': 0.30,
    'Infusion Center': 0.15,
    'Palliative Care': 0.05,
    'Physical Rehabilitation': 0.60,
    'Occupational Therapy': 0.25,
    'Speech Therapy': 0.15,
    'Pathology': 0.05,
    'Laboratory': 0.05,
    'default': 0.10
}

# ICD-10 codes
ICD_CODES = [
    'I10', 'E11.9', 'Z00.00', 'M54.5', 'J00', 'R05', 'F41.9', 'N28.1',
    'Z01.419', 'Z51.11', 'I25.10', 'E78.5', 'Z95.1', 'M79.3', 'R10.9',
    'F32.9', 'N39.0', 'Z12.31', 'Z79.899', 'I48.91', 'G43.909', 'K21.9',
    'Z98.891', 'M17.9', 'R51', 'F33.9', 'N18.9', 'Z23', 'Z87.440', 'I50.9',
    'J18.9', 'A41.9', 'J44.9', 'L03.90', 'I63.9', 'R55', 'J11.00',
    'W19.XXXA', 'V89.2'
]

# Medications
MEDICATIONS = [
    'Lisinopril', 'Metformin', 'Amlodipine', 'Omeprazole', 'Simvastatin', 'Losartan', 'Albuterol',
    'Gabapentin', 'Sertraline', 'Furosemide', 'Fluticasone', 'Prednisone', 'Warfarin', 'Levothyroxine',
    'Hydrochlorothiazide', 'Aspirin', 'Atorvastatin', 'Clopidogrel', 'Montelukast', 'Trazodone',
    'Citalopram', 'Duloxetine', 'Escitalopram', 'Bupropion', 'Venlafaxine', 'Quetiapine',
    'Risperidone', 'Olanzapine', 'Lamotrigine', 'Topiramate', 'Ceftriaxone', 'Vancomycin',
    'Piperacillin-Tazobactam', 'Norepinephrine'
]

# Lab tests
LAB_TESTS = [
    'CBC', 'CMP', 'Lipid Panel', 'TSH', 'Hemoglobin A1c', 'Urinalysis', 'Chest X-Ray', 'EKG',
    'PT/INR', 'Vitamin D', 'B12', 'Folate', 'Ferritin', 'Iron', 'TIBC', 'Liver Function',
    'Kidney Function', 'Electrolytes', 'Glucose', 'Cortisol', 'T3/T4', 'PSA', 'CA-125',
    'CEA', 'AFP', 'Beta-HCG', 'Troponin', 'CK-MB', 'BNP', 'D-Dimer', 'Lactic Acid',
    'Blood Culture', 'ABG', 'CT Head'
]

# Diagnosis details
DIAGNOSIS_DETAILS = {
    'I10': {'description': 'Essential (primary) hypertension', 'complaints': ['Headache', 'Dizziness']},
    'E11.9': {'description': 'Type 2 diabetes mellitus', 'complaints': ['Frequent urination', 'Increased thirst']},
    'J00': {'description': 'Acute nasopharyngitis (common cold)', 'complaints': ['Sore throat', 'Runny nose']},
    'I25.10': {'description': 'Atherosclerotic heart disease', 'complaints': ['Chest pain', 'Shortness of breath']},
    'I50.9': {'description': 'Heart failure, unspecified', 'complaints': ['Shortness of breath', 'Fatigue']},
    'J18.9': {'description': 'Pneumonia, unspecified', 'complaints': ['Fever', 'Cough', 'Shortness of breath']},
    'J11.00': {'description': 'Influenza with pneumonia', 'complaints': ['Fever', 'Cough', 'Body aches']},
    'A41.9': {'description': 'Sepsis, unspecified', 'complaints': ['Fever', 'Hypotension', 'Altered mental status']},
}

# Diagnosis to medications mapping
DIAGNOSIS_TO_MEDS = {
    'I10': ['Lisinopril', 'Amlodipine'], 'E11.9': ['Metformin', 'Insulin'],
    'J00': ['Albuterol'], 'N39.0': ['Ciprofloxacin'],
    'J18.9': ['Ceftriaxone', 'Azithromycin'], 'I50.9': ['Furosemide', 'Lisinopril']
}

# Diagnosis to labs mapping
DIAGNOSIS_TO_LABS = {
    'I10': ['CMP', 'Lipid Panel'], 'E11.9': ['Hemoglobin A1c', 'Glucose'],
    'I25.10': ['Troponin', 'EKG'], 'J18.9': ['CBC', 'Chest X-Ray'],
    'A41.9': ['CBC', 'Lactic Acid', 'Blood Culture']
}

# Weather impacts
WEATHER_IMPACT = {
    'rain': {'flu_multiplier': 1.5, 'fall_multiplier': 1.8, 'accident_multiplier': 1.3},
    'snow': {'flu_multiplier': 2.0, 'fall_multiplier': 2.5, 'accident_multiplier': 2.0},
    'clear': {'flu_multiplier': 1.0, 'fall_multiplier': 1.0, 'accident_multiplier': 1.0},
    'cloudy': {'flu_multiplier': 1.1, 'fall_multiplier': 1.05, 'accident_multiplier': 1.05}
}

# Weather cache
WEATHER_CACHE = {}
WEATHER_CACHE_FILE = 'weather_cache.json'

def load_weather_cache():
    """Load weather cache from disk if it exists."""
    global WEATHER_CACHE
    if os.path.exists(WEATHER_CACHE_FILE):
        try:
            with open(WEATHER_CACHE_FILE, 'r') as f:
                WEATHER_CACHE = json.load(f)
        except Exception as e:
            print(f"Warning: could not load weather cache: {e}")

def save_weather_cache():
    """Save weather cache to disk for persistence across runs."""
    try:
        with open(WEATHER_CACHE_FILE, 'w') as f:
            json.dump(WEATHER_CACHE, f, indent=2)
    except Exception as e:
        print(f"Warning: could not save weather cache: {e}")

load_weather_cache()

# Create database engine
try:
    engine = create_engine(f'mssql+pyodbc:///?odbc_connect={CONNECTION_STRING}') if CONNECTION_STRING else None
except Exception as e:
    print(f'Warning: Could not create engine: {e}')
    engine = None

print('✓ Configuration and constants loaded')

In [ ]:
# ============================================================================
# HELPER FUNCTIONS - Hospital Specialization & Seasonality
# ============================================================================

HOSPITAL_MONTHLY = {
    'general': GENERAL_MONTHLY,
    'pediatric': PEDIATRIC_MONTHLY,
    'cardiac': CARDIAC_MONTHLY,
    'cancer': CANCER_MONTHLY,
    'rehab': REHAB_MONTHLY,
    'specialty': GENERAL_MONTHLY
}

def get_hospital_monthly_encounters(hospital_specialty, month, base_encounters=300):
    """Calculate encounters for a hospital on a given month."""
    multiplier = HOSPITAL_MONTHLY[hospital_specialty][month - 1]
    return int(base_encounters * multiplier)

def get_specialty_diagnoses(hospital_specialty):
    """Get diagnosis weights appropriate to hospital specialty."""
    diagnoses = {
        'pediatric': {'J00': 0.22, 'H66.001': 0.18, 'J45.9': 0.12, 'S72.1': 0.15, 'J18.9': 0.08},
        'cardiac': {'I21.9': 0.20, 'I21.0': 0.15, 'I20.0': 0.10, 'I50.9': 0.22, 'I48.91': 0.18},
        'cancer': {'C92.0': 0.12, 'C91.0': 0.08, 'C82.9': 0.10, 'C85.9': 0.08, 'C80.1': 0.18},
        'rehab': {'I69.3': 0.25, 'M17.11': 0.15, 'M16.1': 0.12, 'Z54': 0.20},
        'general': {'I10': 0.15, 'E11.9': 0.12, 'J00': 0.10, 'I21.9': 0.08, 'I50.9': 0.07},
    }
    return diagnoses.get(hospital_specialty, diagnoses['general'])

def get_admission_rate(hospital_specialty):
    """Get baseline admission rate by hospital specialty."""
    rates = {'pediatric': 0.12, 'cardiac': 0.72, 'cancer': 0.96, 'rehab': 0.85, 'general': 0.20, 'specialty': 0.15}
    return rates.get(hospital_specialty, 0.20)

def get_average_los(hospital_specialty):
    """Get average length of stay (days) by hospital specialty."""
    los = {'pediatric': 1.2, 'cardiac': 3.5, 'cancer': 7.4, 'rehab': 21.0, 'general': 3.8, 'specialty': 2.0}
    return los.get(hospital_specialty, 3.0)

def calculate_er_occupancy_at_hour(hospital_id, hour_of_day, day_of_week):
    """Calculate ER occupancy percentage at a specific hour."""
    base_occupancy = HOURLY_ED_OCCUPANCY[hour_of_day]
    day_mult = DAY_OF_WEEK_OCCUPANCY_MULT.get(day_of_week, 1.00)
    noise = random.uniform(-0.05, 0.05)
    final_occupancy = base_occupancy * day_mult * (1 + noise)
    return min(100, max(0, final_occupancy))

def get_admission_times_by_specialty(hospital_specialty, num_admissions):
    """Generate realistic admission times for a hospital specialty."""
    times = []
    if hospital_specialty == 'cardiac':
        for _ in range(int(num_admissions * 0.35)):
            times.append(random.randint(6, 9))
        for _ in range(int(num_admissions * 0.28)):
            times.append(random.randint(14, 17))
        for _ in range(int(num_admissions * 0.37)):
            times.append(random.randint(0, 23))
    elif hospital_specialty == 'pediatric':
        for _ in range(int(num_admissions * 0.25)):
            times.append(random.randint(8, 10))
        for _ in range(int(num_admissions * 0.20)):
            times.append(random.randint(14, 16))
        for _ in range(int(num_admissions * 0.22)):
            times.append(random.randint(18, 21))
        for _ in range(int(num_admissions * 0.33)):
            times.append(random.randint(0, 23))
    elif hospital_specialty == 'cancer':
        for _ in range(int(num_admissions * 0.70)):
            times.append(random.randint(8, 17))
        for _ in range(int(num_admissions * 0.30)):
            times.append(random.randint(0, 8))
    elif hospital_specialty == 'rehab':
        for _ in range(int(num_admissions * 0.60)):
            times.append(random.randint(7, 11))
        for _ in range(int(num_admissions * 0.30)):
            times.append(random.randint(13, 16))
        for _ in range(int(num_admissions * 0.10)):
            times.append(random.randint(0, 23))
    else:
        for _ in range(int(num_admissions * 0.35)):
            times.append(random.randint(8, 12))
        for _ in range(int(num_admissions * 0.40)):
            times.append(random.randint(12, 18))
        for _ in range(int(num_admissions * 0.25)):
            times.append(random.randint(18, 23))
    while len(times) < num_admissions:
        times.append(random.randint(0, 23))
    return times[:num_admissions]

def get_payer_for_patient(age, hospital_specialty):
    """Return a payer string sampled from market share adjusted for age and specialty."""
    base = PAYER_MARKET_SHARE.copy()
    if age >= 65:
        base['Medicare'] = base.get('Medicare', 0.36) * 1.5
        base['Commercial'] = base.get('Commercial', 0.35) * 0.8
        base['Medicaid'] = base.get('Medicaid', 0.20) * 0.8
    elif age < 18:
        base['Medicaid'] = base.get('Medicaid', 0.20) * 1.3
        base['Commercial'] = base.get('Commercial', 0.35) * 0.9
    adjust = SPECIALTY_PAYER_ADJUST.get(hospital_specialty, {})
    for k, v in adjust.items():
        base[k] = base.get(k, 0.0) * v
    total = sum(base.values())
    if total <= 0:
        return random.choice(['Commercial', 'Medicare', 'Medicaid', 'Uninsured'])
    probs = {k: v / total for k, v in base.items()}
    r = random.random()
    cum = 0.0
    for k, p in probs.items():
        cum += p
        if r <= cum:
            return k
    return k

def apply_diagnosis_seasonality(icd_code, month, hospital_specialty, weather_condition=None):
    """Return a multiplier for the given diagnosis based on month, specialty, and weather."""
    m = 1.0
    if icd_code.startswith('J11') or icd_code == 'J00':
        if month in [12, 1, 2]:
            m *= 2.0
    if icd_code == 'J45.9' and hospital_specialty == 'pediatric' and month in [11, 12, 1, 2]:
        m *= 1.8
    if icd_code.startswith('I21') or icd_code.startswith('I50'):
        if month in [12, 1, 2]:
            m *= 1.25
    if icd_code.startswith('W') or icd_code.startswith('V'):
        if month in [12, 1, 2]:
            m *= 1.3
        if month in [6, 7, 8]:
            m *= 1.2
    if weather_condition == 'snow' and (icd_code.startswith('W') or icd_code.startswith('V')):
        m *= 1.8
    if weather_condition == 'rain' and icd_code.startswith('V'):
        m *= 1.3
    return m

def sample_diagnosis_for_hospital(hospital_specialty, month, weather_condition=None):
    """Sample a diagnosis ICD code for a hospital specialty applying seasonality and specialty weights."""
    diags = get_specialty_diagnoses(hospital_specialty)
    weighted = {}
    for code, base_w in diags.items():
        mult = apply_diagnosis_seasonality(code, month, hospital_specialty, weather_condition=weather_condition)
        weighted[code] = max(0.0, base_w * mult)
    total = sum(weighted.values())
    if total <= 0:
        return random.choice(list(diags.keys()))
    rnd = random.random() * total
    cum = 0.0
    for code, w in weighted.items():
        cum += w
        if rnd <= cum:
            return code
    return code

print('✓ Helper functions loaded')

In [ ]:
# ============================================================================
# WEATHER & UTILITY FUNCTIONS
# ============================================================================

def get_current_weather(latitude=40.7128, longitude=-74.0060):
    """Fetch current weather conditions."""
    try:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=weather_code"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            weather_code = data['current']['weather_code']
            if weather_code == 0:
                condition = 'clear'
            elif weather_code in [1, 2]:
                condition = 'cloudy'
            elif weather_code in [51, 53, 55, 61, 63, 65, 80, 81, 82]:
                condition = 'rain'
            elif weather_code in [71, 73, 75, 77, 85, 86]:
                condition = 'snow'
            else:
                condition = 'cloudy'
            return condition, WEATHER_IMPACT.get(condition, WEATHER_IMPACT['clear'])
    except Exception:
        pass
    return 'clear', WEATHER_IMPACT['clear']

def get_weather_for_date(date, latitude=40.7128, longitude=-74.0060):
    """Fetch weather for a specific date."""
    d = pd.to_datetime(date).date()
    date_str = d.isoformat()
    if date_str in WEATHER_CACHE:
        condition = WEATHER_CACHE[date_str]
        return condition, WEATHER_IMPACT.get(condition, WEATHER_IMPACT['clear'])
    try:
        url = f"https://archive-api.open-meteo.com/v1/archive?latitude={latitude}&longitude={longitude}&start_date={date_str}&end_date={date_str}&daily=weathercode"
        resp = requests.get(url, timeout=8)
        if resp.status_code == 200:
            data = resp.json()
            weather_codes = data.get('daily', {}).get('weathercode')
            if weather_codes and len(weather_codes) > 0:
                weather_code = weather_codes[0]
                if weather_code == 0:
                    condition = 'clear'
                elif weather_code in [51, 53, 55, 61, 63, 65]:
                    condition = 'rain'
                elif weather_code in [71, 73, 75, 77]:
                    condition = 'snow'
                else:
                    condition = 'cloudy'
                WEATHER_CACHE[date_str] = condition
                return condition, WEATHER_IMPACT.get(condition, WEATHER_IMPACT['clear'])
    except Exception:
        pass
    WEATHER_CACHE[date_str] = 'clear'
    return 'clear', WEATHER_IMPACT['clear']

def get_next_id(table_name, id_col):
    """Return next id to use for inserts by querying the current max in the DB."""
    if not engine:
        return 1
    try:
        q = f"SELECT MAX({id_col}) as max_id FROM {table_name}"
        df = pd.read_sql(q, engine)
        max_id = df.iloc[0]['max_id']
        if pd.isna(max_id) or max_id is None:
            return 1
        return int(max_id) + 1
    except Exception:
        return 1

def get_table_columns(table_name):
    """Get list of column names from a table."""
    if not engine:
        return []
    try:
        inspector = inspect(engine)
        cols = inspector.get_columns(table_name)
        return [c['name'] for c in cols]
    except Exception:
        return []

def get_encounter_date_range():
    """Get min and max encounter dates from DB."""
    if not engine:
        return None, None
    try:
        df = pd.read_sql("SELECT MIN(encounter_date) as min_date, MAX(encounter_date) as max_date FROM encounters", engine)
        min_d = df.iloc[0]['min_date']
        max_d = df.iloc[0]['max_date']
        return pd.to_datetime(min_d).date() if not pd.isna(min_d) else None, pd.to_datetime(max_d).date() if not pd.isna(max_d) else None
    except Exception:
        return None, None

print('✓ Weather and utility functions loaded')

In [ ]:
# ============================================================================
# GENERATOR FUNCTIONS
# ============================================================================

def generate_patients(count):
    """Generate synthetic patient records."""
    rows = []
    for i in range(1, count + 1):
        age = random.randint(18, 89)
        dob = (pd.to_datetime('today') - pd.Timedelta(days=age*365.25)).date()
        rows.append({
            'patient_id': i,
            'mrn': f'MRN{fake.random_int(100000, 999999)}',
            'date_of_birth': dob,
            'age': age,
            'gender': random.choice(['M', 'F']),
            'city': fake.city(),
            'state': fake.state_abbr(),
            'primary_hospital_id': random.choice([h['hospital_id'] for h in HOSPITALS])
        })
    return pd.DataFrame(rows)

def generate_doctors(count):
    """Generate synthetic doctor records."""
    rows = []
    specialties = ['Internal Medicine', 'Cardiology', 'Surgery', 'Pediatrics', 'Oncology', 'Rehabilitation']
    for i in range(1, count + 1):
        rows.append({
            'provider_id': i,
            'first_name': fake.first_name(),
            'last_name': fake.last_name(),
            'specialty': random.choice(specialties),
            'hospital_id': random.choice([h['hospital_id'] for h in HOSPITALS]),
            'npi': f'NPI{fake.random_int(1000000000, 9999999999)}'
        })
    return pd.DataFrame(rows)

def generate_encounters(encounter_count, min_date=None, end_date=None, patient_start=1, patient_count=100, provider_start=1, provider_count=10, hospital_id=None):
    """Generate synthetic encounter records with department and payer assignment."""
    if min_date is None:
        min_date = datetime.now() - timedelta(days=30)
    if end_date is None:
        end_date = datetime.now()
    min_date = pd.to_datetime(min_date)
    end_date = pd.to_datetime(end_date)
    rows = []
    for i in range(1, encounter_count + 1):
        enc_date = min_date + (end_date - min_date) * random.random()
        enc_date = enc_date.date()
        
        # Select hospital
        if hospital_id:
            hospital = next((h for h in HOSPITALS if h['hospital_id'] == hospital_id), HOSPITALS[0])
        else:
            hospital = random.choice(HOSPITALS)
        
        # Select department with realistic demand distribution
        hosp_depts = [d for d in DEPARTMENTS if d['hospital_id'] == hospital['hospital_id']]
        if hosp_depts:
            dept_weights = [DEPARTMENT_DEMAND_MULTIPLIERS.get(d['name'], DEPARTMENT_DEMAND_MULTIPLIERS.get(d['specialty_type'], 1.0)) for d in hosp_depts]
            department = random.choices(hosp_depts, weights=dept_weights)[0]
        else:
            department = {'department_id': 1, 'name': 'General'}
        
        # Assign payer based on hospital specialty
        specialty = hospital.get('specialty', 'general')
        patient_age = random.randint(0, 90)
        payer_type = get_payer_for_patient(patient_age, specialty)
        
        encounter_type = random.choice(['Emergency', 'Urgent', 'Inpatient'])
        if encounter_type == 'Emergency':
            admission_time = random.randint(0, 23)
            los_hours = random.randint(2, 12)
        elif encounter_type == 'Urgent':
            admission_time = random.randint(7, 18)
            los_hours = random.randint(1, 4)
        else:
            admission_time = random.randint(7, 12)
            los_hours = random.randint(24, 168)
        
        rows.append({
            'encounter_id': i,
            'patient_id': random.randint(patient_start, patient_start + patient_count - 1),
            'provider_id': random.randint(provider_start, provider_start + provider_count - 1),
            'hospital_id': hospital['hospital_id'],
            'department_id': department['department_id'],
            'encounter_type': encounter_type,
            'encounter_date': enc_date,
            'admission_time': admission_time,
            'payer_type': payer_type,
            'los_hours': los_hours
        })
    return pd.DataFrame(rows)

def generate_diagnoses(diagnosis_count, min_date=None, **kwargs):
    """Generate synthetic diagnosis records."""
    if min_date is None:
        min_date = datetime.now() - timedelta(days=30)
    rows = []
    for i in range(1, diagnosis_count + 1):
        icd_code = random.choice(ICD_CODES)
        details = DIAGNOSIS_DETAILS.get(icd_code, {'description': 'Unknown diagnosis', 'complaints': ['Unknown']})
        rows.append({
            'diagnosis_id': i,
            'encounter_id': random.randint(1, max(1, diagnosis_count // 2)),
            'patient_id': random.randint(1, max(1, diagnosis_count // 4)),
            'icd_code': icd_code,
            'description': details['description'],
            'is_primary': random.choice([True, False]),
            'onset_date': pd.to_datetime(min_date).date(),
            'status': random.choice(['Active', 'Resolved', 'Chronic'])
        })
    return pd.DataFrame(rows)

def generate_diagnoses_weather_aware(diagnosis_count, min_date=None, weather_condition='clear', weather_impacts=None, **kwargs):
    """Generate diagnoses with weather awareness."""
    if weather_impacts is None:
        weather_impacts = WEATHER_IMPACT.get(weather_condition, WEATHER_IMPACT['clear'])
    if min_date is None:
        min_date = datetime.now() - timedelta(days=30)
    rows = []
    for i in range(1, diagnosis_count + 1):
        icd_code = random.choice(ICD_CODES)
        details = DIAGNOSIS_DETAILS.get(icd_code, {'description': 'Unknown diagnosis', 'complaints': ['Unknown']})
        rows.append({
            'diagnosis_id': i,
            'encounter_id': random.randint(1, max(1, diagnosis_count // 2)),
            'patient_id': random.randint(1, max(1, diagnosis_count // 4)),
            'icd_code': icd_code,
            'description': details['description'],
            'is_primary': random.choice([True, False]),
            'onset_date': pd.to_datetime(min_date).date(),
            'status': random.choice(['Active', 'Resolved', 'Chronic'])
        })
    return pd.DataFrame(rows)

def generate_procedures(procedure_count, min_date=None, end_date=None, encounters_df=None, **kwargs):
    """Generate synthetic procedure records."""
    if min_date is None:
        min_date = datetime.now() - timedelta(days=30)
    if end_date is None:
        end_date = datetime.now()
    rows = []
    for i in range(1, procedure_count + 1):
        proc_date = pd.to_datetime(min_date) + (pd.to_datetime(end_date) - pd.to_datetime(min_date)) * random.random()
        procedure = random.choice(list(PROCEDURE_WEIGHTS.keys()))
        encounter_id = 1
        if encounters_df is not None and len(encounters_df) > 0:
            encounter_id = random.choice(encounters_df['encounter_id'].tolist())
        rows.append({
            'procedure_id': i,
            'encounter_id': encounter_id,
            'patient_id': random.randint(1, max(1, procedure_count // 3)),
            'procedure_name': procedure,
            'procedure_date': proc_date.date(),
            'cost': round(random.uniform(500, 5000), 2)
        })
    return pd.DataFrame(rows)

def generate_medications(medication_count, min_date=None, end_date=None, encounters_df=None, diagnoses_df=None, **kwargs):
    """Generate synthetic medication records."""
    if min_date is None:
        min_date = datetime.now() - timedelta(days=30)
    if end_date is None:
        end_date = datetime.now()
    rows = []
    for i in range(1, medication_count + 1):
        med_date = pd.to_datetime(min_date) + (pd.to_datetime(end_date) - pd.to_datetime(min_date)) * random.random()
        medication = random.choice(MEDICATIONS)
        encounter_id = 1
        if encounters_df is not None and len(encounters_df) > 0:
            encounter_id = random.choice(encounters_df['encounter_id'].tolist())
        rows.append({
            'medication_id': i,
            'encounter_id': encounter_id,
            'patient_id': random.randint(1, max(1, medication_count // 3)),
            'medication_name': medication,
            'dosage': f"{random.choice([250, 500, 750, 1000])} mg",
            'route': random.choice(['PO', 'IV', 'IM', 'Topical']),
            'frequency': random.choice(['Once daily', 'Twice daily', 'Three times daily', 'As needed']),
            'start_date': med_date.date()
        })
    return pd.DataFrame(rows)

def generate_labs(lab_count, min_date=None, end_date=None, encounters_df=None, diagnoses_df=None, **kwargs):
    """Generate synthetic lab results."""
    if min_date is None:
        min_date = datetime.now() - timedelta(days=30)
    if end_date is None:
        end_date = datetime.now()
    rows = []
    for i in range(1, lab_count + 1):
        lab_date = pd.to_datetime(min_date) + (pd.to_datetime(end_date) - pd.to_datetime(min_date)) * random.random()
        lab_test = random.choice(LAB_TESTS)
        encounter_id = 1
        if encounters_df is not None and len(encounters_df) > 0:
            encounter_id = random.choice(encounters_df['encounter_id'].tolist())
        rows.append({
            'lab_id': i,
            'encounter_id': encounter_id,
            'patient_id': random.randint(1, max(1, lab_count // 3)),
            'lab_test': lab_test,
            'result_value': round(random.uniform(50, 150), 2),
            'unit': random.choice(['mg/dL', 'mEq/L', '%', 'cells/mcL']),
            'result_date': lab_date.date()
        })
    return pd.DataFrame(rows)

def generate_insurance_for_patients(patients_df, min_date=None):
    """Generate insurance records for patients."""
    if min_date is None:
        min_date = datetime.now()
    rows = []
    for idx, patient in patients_df.iterrows():
        payer = get_payer_for_patient(patient['age'], 'general')
        rows.append({
            'insurance_id': idx + 1,
            'patient_id': patient['patient_id'],
            'payer': payer,
            'policy_number': f'POL{fake.random_int(100000, 999999)}',
            'group_number': f'GRP{fake.random_int(10000, 99999)}',
            'effective_date': pd.to_datetime(min_date).date(),
            'termination_date': None
        })
    return pd.DataFrame(rows)

def generate_billing(billing_count, min_date=None):
    """Generate billing records."""
    if min_date is None:
        min_date = datetime.now() - timedelta(days=30)
    rows = []
    for i in range(1, billing_count + 1):
        bill_date = pd.to_datetime(min_date) + timedelta(days=random.randint(0, 30))
        rows.append({
            'billing_id': i,
            'encounter_id': random.randint(1, max(1, billing_count // 2)),
            'patient_id': random.randint(1, max(1, billing_count // 3)),
            'total_charges': round(random.uniform(1000, 50000), 2),
            'insurance_paid': round(random.uniform(500, 40000), 2),
            'patient_responsibility': round(random.uniform(100, 5000), 2),
            'billing_date': bill_date.date()
        })
    return pd.DataFrame(rows)

def generate_date_dimension(start_date, end_date):
    """Generate a comprehensive date dimension table."""
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    dates = pd.date_range(start=start, end=end, freq='D')
    rows = []
    for d in dates:
        iso = d.isocalendar()
        fiscal_start_month = 1
        fiscal_year = d.year if d.month >= fiscal_start_month else d.year - 1
        rows.append({
            'date': d.date(),
            'date_key': int(d.strftime('%Y%m%d')),
            'yyyy': d.year,
            'mm': d.month,
            'mm_name': d.strftime('%B'),
            'mm_short': d.strftime('%b'),
            'dd': d.day,
            'day_of_week': d.weekday(),
            'day_of_week_name': d.strftime('%A'),
            'is_weekend': 1 if d.weekday() >= 5 else 0,
            'quarter': (d.month - 1) // 3 + 1,
            'iso_year': iso.year,
            'iso_week': int(iso.week),
            'day_of_year': int(d.strftime('%j')),
            'week_of_year': int(d.strftime('%U')),
            'fiscal_year': fiscal_year,
            'fiscal_month': ((d.month - fiscal_start_month) % 12) + 1,
            'mm_dd_yyyy': d.strftime('%m-%d-%Y'),
            'dd_mm_yyyy': d.strftime('%d-%m-%Y'),
            'yyyy_mm_dd': d.strftime('%Y-%m-%d')
        })
    return pd.DataFrame(rows)

print('✓ Generator functions loaded')

In [ ]:
# ============================================================================
# MAIN GENERATOR ORCHESTRATOR & DATABASE FUNCTIONS
# ============================================================================

def get_table_columns(table_name):
    """Get list of column names from a table."""
    if not engine:
        return []
    try:
        inspector_obj = inspect(engine)
        cols = inspector_obj.get_columns(table_name)
        return [c['name'] for c in cols]
    except Exception:
        return []

def align_df_to_table(df, table_name):
    """Return a dataframe containing only columns that exist in target table."""
    cols = get_table_columns(table_name)
    if not cols:
        return df
    common = [c for c in df.columns if c in cols]
    if not common:
        return df
    return df[common]

def generate_admissions_for_day_specialty_aware(hospitals_df, encounters_df, patients_df, departments_df, date, start_admission_id=1):
    """Generate admissions for inpatient encounters on the given date, using hospital-specific admission rates and timing."""
    rows = []
    if 'encounter_type' in encounters_df.columns:
        inpatients = encounters_df[encounters_df['encounter_type'] == 'Inpatient']
    else:
        inpatients = pd.DataFrame()
    
    aid = start_admission_id
    hospital_lookup = {h['hospital_id']: h for h in HOSPITALS}
    
    for _, enc in inpatients.iterrows():
        hid = int(enc.get('hospital_id', random.choice(hospitals_df['hospital_id'].tolist())))
        hospital = hospital_lookup.get(hid, {'specialty': 'general'})
        specialty = hospital.get('specialty', 'general')
        admission_prob = get_admission_rate(specialty)
        
        if random.random() > admission_prob:
            continue
        
        pid = int(enc.get('patient_id', random.randint(1, max(1, len(patients_df)))))
        dept_id = enc.get('provider_id')
        
        ed = departments_df[(departments_df['hospital_id'] == hid) & (departments_df['name'].str.lower().str.contains('emergency', na=False))]
        if not ed.empty:
            dept_id = int(ed.iloc[0]['department_id'])
        
        admit_hours = get_admission_times_by_specialty(specialty, num_admissions=1)
        admit_hour = admit_hours[0] if admit_hours else random.randint(0, 23)
        admit_dt = pd.to_datetime(date) + pd.Timedelta(hours=admit_hour, minutes=random.randint(0, 59))
        
        avg_los = get_average_los(specialty)
        if random.random() < 0.60:
            los_days = max(0, int(avg_los + random.gauss(0, avg_los * 0.3)))
            discharge_dt = admit_dt + pd.Timedelta(days=los_days, hours=random.randint(7, 17))
        else:
            discharge_dt = None
        
        row = {
            'admission_id': aid,
            'patient_id': pid,
            'hospital_id': hid,
            'department_id': int(dept_id) if dept_id is not None else None,
            'encounter_id': int(enc.get('encounter_id', -1)),
            'admit_datetime': admit_dt,
            'discharge_datetime': discharge_dt
        }
        rows.append(row)
        aid += 1
    
    return pd.DataFrame(rows)

def main(rebuild_start=None, export_parquet=False):
    """Main generator orchestrator."""
    if not engine:
        print('❌ No database engine available. Set CONNECTION_STRING first.')
        return
    
    if rebuild_start:
        try:
            start_date = datetime.strptime(rebuild_start, "%Y-%m-%d").date()
        except Exception as e:
            print(f"❌ Invalid rebuild start date: {rebuild_start}. Use YYYY-MM-DD.")
            return
        end_date = datetime.now().date()
        days = (end_date - start_date).days + 1
        print(f"\n🔄 FULL REBUILD: {start_date} → {end_date} ({days} days)")
    else:
        # INCREMENTAL MODE: Find last date in encounters table
        print("\n📊 INCREMENTAL MODE")
        try:
            result = pd.read_sql("SELECT MAX(encounter_date) as max_date FROM encounters", engine)
            last_date = result.iloc[0]['max_date']
            if pd.isna(last_date) or last_date is None:
                print("❌ No existing encounter data found. Use full rebuild mode instead.")
                print("   Set rebuild_start='2025-01-01' (or your desired start date)")
                return
            last_date = pd.to_datetime(last_date).date()
            start_date = last_date + timedelta(days=1)
            end_date = datetime.now().date()
            
            if start_date > end_date:
                print(f"✓ Data is current! Last encounter date: {last_date}")
                print("  No new data to generate.")
                return
            
            days = (end_date - start_date).days + 1
            print(f"📅 Last encounter in DB: {last_date}")
            print(f"📅 Generating catchup: {start_date} → {end_date} ({days} days)")
        except Exception as e:
            print(f"❌ Error checking encounters table: {type(e).__name__}: {e}")
            print("   Make sure the encounters table exists. Use full rebuild to create it.")
            return
        
        # Generate date dimension
        print("  Generating date dimension...")
        date_dim_df = generate_date_dimension(start_date, end_date)
        date_dim_df.to_sql('date_dim', engine, if_exists='replace', index=False)
        
        # Configuration
        patient_count = 50
        doctor_count = 5
        procedure_count = 100
        encounter_count = 150
        diagnosis_count = 200
        medication_count = 125
        lab_count = 175
        insurance_count = 50
        billing_count = 150
        
        next_ids = {
            'patient': 1, 'provider': 1, 'encounter': 1, 'procedure': 1,
            'diagnosis': 1, 'medication': 1, 'lab': 1, 'insurance': 1,
            'billing': 1, 'admission': 1
        }
        
        # For incremental, get next IDs from DB to avoid collisions
        if rebuild_start is None:
            # Incremental mode - get max IDs from tables
            try:
                for key, table_name, id_col in [
                    ('patient', 'patients', 'patient_id'),
                    ('provider', 'doctors', 'provider_id'),
                    ('encounter', 'encounters', 'encounter_id'),
                    ('procedure', 'procedures', 'procedure_id'),
                    ('diagnosis', 'diagnoses', 'diagnosis_id'),
                    ('medication', 'medications', 'medication_id'),
                    ('lab', 'labs', 'lab_id'),
                    ('insurance', 'insurance', 'insurance_id'),
                    ('billing', 'billing', 'billing_id'),
                    ('admission', 'admissions', 'admission_id')
                ]:
                    result = pd.read_sql(f"SELECT MAX({id_col}) as max_id FROM {table_name}", engine)
                    max_id = result.iloc[0]['max_id']
                    if not pd.isna(max_id) and max_id is not None:
                        next_ids[key] = int(max_id) + 1
            except Exception as e:
                print(f"  ⚠ Could not fetch max IDs (using defaults): {e}")
        
        first = (rebuild_start is not None)  # Only replace tables on full rebuild
        for day_offset in range(days):
            day = start_date + timedelta(days=day_offset)
            if day_offset % 10 == 0:
                print(f"  Generating data for {day}...")
            
            hospitals_df = pd.DataFrame(HOSPITALS)
            departments_df = pd.DataFrame(DEPARTMENTS)
            
            # Generate base entities
            patients_df = generate_patients(patient_count)
            patients_df['patient_id'] = range(next_ids['patient'], next_ids['patient'] + len(patients_df))
            
            doctors_df = generate_doctors(doctor_count)
            doctors_df['provider_id'] = range(next_ids['provider'], next_ids['provider'] + len(doctors_df))
            
            # Generate encounters per hospital
            encounters_df = pd.DataFrame()
            diagnoses_df = pd.DataFrame()
            month = day.month
            
            for hospital in HOSPITALS:
                hospital_id = hospital['hospital_id']
                specialty = hospital.get('specialty', 'general')
                
                hosp_encounter_count = get_hospital_monthly_encounters(specialty, month, base_encounters=150)
                
                # Scale by hospital size (bed_count) with realistic variation (5-30%)
                total_beds = sum(h['bed_count'] for h in HOSPITALS)
                bed_ratio = hospital['bed_count'] / total_beds
                
                # Apply bed-based distribution with random variation (5-30%)
                variation = random.uniform(0.85, 1.30)  # +/- 15% to 30% variation
                hosp_encounter_count = max(1, int(hosp_encounter_count * bed_ratio * 10 * variation))
                
                hosp_encounters = generate_encounters(
                    hosp_encounter_count,
                    min_date=day,
                    end_date=day,
                    patient_start=next_ids['patient'],
                    patient_count=patient_count,
                    provider_start=next_ids['provider'],
                    provider_count=doctor_count,
                    hospital_id=hospital_id
                )
                hosp_encounters['encounter_id'] = range(next_ids['encounter'], next_ids['encounter'] + len(hosp_encounters))
                next_ids['encounter'] += len(hosp_encounters)
                encounters_df = pd.concat([encounters_df, hosp_encounters], ignore_index=True)
                
                # Weather-aware diagnoses
                w_cond, w_imp = get_weather_for_date(day)
                specialty_diags = get_specialty_diagnoses(specialty)
                hosp_diagnosis_count = int(diagnosis_count * (hosp_encounter_count / encounter_count)) if encounter_count > 0 else 0
                hosp_diagnoses = []
                for _ in range(hosp_diagnosis_count):
                    icd_code = sample_diagnosis_for_hospital(specialty, day.month, weather_condition=w_cond)
                    details = DIAGNOSIS_DETAILS.get(icd_code, {'description': 'Unknown', 'complaints': ['Unknown']})
                    diag = {
                        'encounter_id': random.choice(hosp_encounters['encounter_id'].tolist()) if len(hosp_encounters) > 0 else 1,
                        'patient_id': random.randint(next_ids['patient'], max(next_ids['patient'], next_ids['patient'] + patient_count - 1)),
                        'icd_code': icd_code,
                        'description': details['description'],
                        'onset_date': day,
                        'status': random.choice(['Active', 'Resolved', 'Chronic'])
                    }
                    hosp_diagnoses.append(diag)
                
                hosp_diag_df = pd.DataFrame(hosp_diagnoses)
                if not hosp_diag_df.empty:
                    hosp_diag_df['diagnosis_id'] = range(next_ids['diagnosis'], next_ids['diagnosis'] + len(hosp_diag_df))
                    next_ids['diagnosis'] += len(hosp_diag_df)
                    diagnoses_df = pd.concat([diagnoses_df, hosp_diag_df], ignore_index=True)
            
            # Procedures, medications, labs
            procedures_df = generate_procedures(procedure_count, min_date=day, end_date=day, encounters_df=encounters_df)
            if not procedures_df.empty:
                procedures_df['procedure_id'] = range(next_ids['procedure'], next_ids['procedure'] + len(procedures_df))
                next_ids['procedure'] += len(procedures_df)
            
            medications_df = generate_medications(medication_count, min_date=day, end_date=day, encounters_df=encounters_df, diagnoses_df=diagnoses_df)
            medications_df['medication_id'] = range(next_ids['medication'], next_ids['medication'] + len(medications_df))
            
            labs_df = generate_labs(lab_count, min_date=day, end_date=day, encounters_df=encounters_df, diagnoses_df=diagnoses_df)
            labs_df['lab_id'] = range(next_ids['lab'], next_ids['lab'] + len(labs_df))
            
            insurance_df = generate_insurance_for_patients(patients_df, min_date=day)
            insurance_df['insurance_id'] = range(next_ids['insurance'], next_ids['insurance'] + len(insurance_df))
            
            billing_df = generate_billing(billing_count, min_date=day)
            billing_df['billing_id'] = range(next_ids['billing'], next_ids['billing'] + len(billing_df))
            
            admissions_df = generate_admissions_for_day_specialty_aware(hospitals_df, encounters_df, patients_df, departments_df, day, start_admission_id=next_ids['admission'])
            next_ids['admission'] += len(admissions_df)
            
            # Insert into DB
            try:
                if first:
                    hospitals_df.to_sql('hospitals', engine, if_exists='replace', index=False)
                    departments_df.to_sql('departments', engine, if_exists='replace', index=False)
                    patients_df.to_sql('patients', engine, if_exists='replace', index=False)
                    doctors_df.to_sql('doctors', engine, if_exists='replace', index=False)
                    if not procedures_df.empty:
                        procedures_df.to_sql('procedures', engine, if_exists='replace', index=False)
                    encounters_df.to_sql('encounters', engine, if_exists='replace', index=False)
                    if not diagnoses_df.empty:
                        diagnoses_df.to_sql('diagnoses', engine, if_exists='replace', index=False)
                    medications_df.to_sql('medications', engine, if_exists='replace', index=False)
                    labs_df.to_sql('labs', engine, if_exists='replace', index=False)
                    insurance_df.to_sql('insurance', engine, if_exists='replace', index=False)
                    billing_df.to_sql('billing', engine, if_exists='replace', index=False)
                    if not admissions_df.empty:
                        admissions_df.to_sql('admissions', engine, if_exists='replace', index=False)
                    first = False
                else:
                    patients_df.to_sql('patients', engine, if_exists='append', index=False)
                    doctors_df.to_sql('doctors', engine, if_exists='append', index=False)
                    if not procedures_df.empty:
                        procedures_df.to_sql('procedures', engine, if_exists='append', index=False)
                    encounters_df.to_sql('encounters', engine, if_exists='append', index=False)
                    if not diagnoses_df.empty:
                        diagnoses_df.to_sql('diagnoses', engine, if_exists='append', index=False)
                    medications_df.to_sql('medications', engine, if_exists='append', index=False)
                    labs_df.to_sql('labs', engine, if_exists='append', index=False)
                    insurance_df.to_sql('insurance', engine, if_exists='append', index=False)
                    billing_df.to_sql('billing', engine, if_exists='append', index=False)
                    if not admissions_df.empty:
                        admissions_df.to_sql('admissions', engine, if_exists='append', index=False)
                
                next_ids['patient'] += len(patients_df)
                next_ids['provider'] += len(doctors_df)
            except Exception as e:
                print(f"  Error inserting data for {day}: {e}")
        
        print(f"\n✅ Data generation complete: {days} days generated")
        save_weather_cache()

print('✓ Main orchestrator function loaded')

## Cell 4: Run the Generator

**Edit the rebuild_start date below before running this cell.**

In [ ]:
# ============================================================================
# RUN THE GENERATOR (Choose one run type by uncommenting the block you want)
# ============================================================================

# --- OPTION A: Full rebuild (default) ---
# This will generate from `rebuild_start` -> today and REPLACE target tables in the DB.
rebuild_start = '2025-01-01'      # Start date (YYYY-MM-DD format)
export_parquet = False              # Set to True to also export Parquet files

# --- OPTION B: Incremental append (uncomment to use) ---
# This will generate incremental data and append to existing tables.
# To run incremental: uncomment the two lines below and comment out Option A lines above.
# rebuild_start = None
# export_parquet = False

# --- OPTION C: Export Parquet only (no DB writes) ---
# Set export_parquet True to write Parquet files to 'data/' instead of DB.
# Uncomment below to enable export-only mode (and optionally set rebuild_start).
# rebuild_start = '2025-01-01'
# export_parquet = True

# --- OTHER FLAGS ---
# To clear the weather cache before running, uncomment the next line:
# WEATHER_CACHE.clear(); import os;
# if os.path.exists(WEATHER_CACHE_FILE): os.remove(WEATHER_CACHE_FILE)

print(f'\n╔═══════════════════════════════════════════════════════════════╗')
print(f'║ Healthcare Data Generator - Selected Run                      ║')
print(f'╚═══════════════════════════════════════════════════════════════╝')
if rebuild_start:
    print(f'\n📅 Mode: FULL REBUILD')
    print(f'📅 Start Date: {rebuild_start}')
else:
    print(f'\n📅 Mode: INCREMENTAL (will start from last date in encounters table)')
print(f'📅 End Date:   {datetime.now().date()}')
print(f'💾 Export Parquet: {export_parquet}')
print(f'🏥 Hospitals: {len(HOSPITALS)}')
print(f'🔬 Departments: {len(DEPARTMENTS)}\n')

if 'rebuild_start' in globals() and rebuild_start is None:
    # User selected incremental append mode
    try:
        main(rebuild_start=None, export_parquet=export_parquet)
    except Exception as e:
        print(f'❌ Generator error (incremental): {type(e).__name__}: {e}')
        import traceback
        traceback.print_exc()
elif 'rebuild_start' in globals() and rebuild_start:
    # Full rebuild selected
    if not CONNECTION_STRING and not export_parquet:
        print('❌ ERROR: No CONNECTION_STRING set. Cannot run generator against DB.')
    else:
        try:
            main(rebuild_start=rebuild_start, export_parquet=export_parquet)
        except Exception as e:
            print(f'❌ Generator error (full rebuild): {type(e).__name__}: {e}')
            import traceback
            traceback.print_exc()
else:
    print('⚠ No valid run option selected. Edit this cell to uncomment a run option.')

## Cell 5: Validation & Results

In [ ]:
# ============================================================================
# POST-RUN VALIDATION: Check row counts
# ============================================================================

if not engine:
    print('❌ No database engine available.')
else:
    print('\n📊 Row Counts in Healthcare ODS Tables:')
    print('─' * 60)
    
    tables_to_check = [
        'patients', 'doctors', 'encounters', 'admissions', 'diagnoses',
        'labs', 'medications', 'procedures', 'insurance', 'billing'
    ]
    
    total_rows = 0
    for table in tables_to_check:
        try:
            count_df = pd.read_sql(f'SELECT COUNT(*) as cnt FROM {table}', engine)
            count = int(count_df.iloc[0]['cnt'])
            total_rows += count
            print(f'  {table:20s}: {count:>10,} rows')
        except Exception as e:
            print(f'  {table:20s}: ERROR - {type(e).__name__}')
    
    print('─' * 60)
    print(f'  Total:               {total_rows:>10,} rows')
    print('\n✅ Data generation validation complete!')